# oyLabImaging Spatial Statistics Pipeline
## Single-Timepoint Example

This notebook demonstrates how to extract, calculate, and visualize spatiotemporal interactions from segmented microscopy data. It covers:
1. Global Spatial Autocorrelation (Moran's I, Geary's C)
2. Local Spatial Hotspot Detection (Local Moran's I)
3. Spatial Expression Mapping
4. Categorical Neighborhood Enrichment
5. Interactive Napari Visualization (Stacked Metric Layers)

In [ ]:
%load_ext autoreload
%autoreload 2
%gui qt
%matplotlib inline

import sys
import os
import dill
import numpy as np
import pandas as pd
import anndata as ad

# Point to our modified package directory
sys.path.insert(0, "../oyLabImaging")
from oyLabImaging import Metadata
from oyLabImaging.Processing.Results import results

### 1. Data Loading & Environment Setup

In [ ]:
# Define the path to the dataset
data_path = '/bigstore/pirlo/Images2025/Sanne/2025.11.23_VME221_IFNdecoy_B18Rtransient/pSTAT1_cntrls_1/'
print("Loading results.pickle...")

# Load the master results object
with open(os.path.join(data_path, "results.pickle"), "rb") as f:
    R = dill.load(f)
R.pth = data_path

# Load the individual Position files (PosLbls) to get the single-cell data
for pos_name in R.PosNames:
    pkl_file = os.path.join(data_path, "PosLbls", f"{pos_name}.pkl")
    if os.path.exists(pkl_file):
        with open(pkl_file, "rb") as f:
            P = dill.load(f)
        P.pth = data_path
        R.PosLbls[pos_name] = P

# Grab the positions we want to compare
test_positions = ['B6-Site_0', 'B5-Site_0']

print(f"✓ Loaded successfully")
print(f"  Positions we are testing: {test_positions}")
print(f"  Timepoints found: {len(R.frames)}")
print(f"  Exact Channels found: {list(R.channels)}")

### 2. Discrete Cell Classification (Quadrant Gating)
Neighborhood Enrichment is designed for discrete cell types. We mimic flow-cytometry "Quadrant Gating" by finding the top 10% of expressors for each marker and categorizing every cell.

In [ ]:
for pos in test_positions:
    # 1. Find the global 90th percentile for the markers 
    all_red = np.concatenate([R.PosLbls[pos].mean('Red')[t] for t in range(len(R.frames)) if R.PosLbls[pos].num[t] > 0])
    all_farred = np.concatenate([R.PosLbls[pos].mean('FarRed')[t] for t in range(len(R.frames)) if R.PosLbls[pos].num[t] > 0])
    
    red_90th = np.percentile(all_red, 90)
    farred_90th = np.percentile(all_farred, 90)

    # 2. Classify each cell based on these biological thresholds
    for t in range(len(R.frames)):
        if R.PosLbls[pos].num[t] > 0:
            r_vals = R.PosLbls[pos].mean('Red')[t]
            fr_vals = R.PosLbls[pos].mean('FarRed')[t]
            
            states = []
            for r, fr in zip(r_vals, fr_vals):
                if r > red_90th and fr > farred_90th:
                    states.append('Double+')    # High Red, High FarRed
                elif r > red_90th:
                    states.append('Red+')       # High Red only
                elif fr > farred_90th:
                    states.append('FarRed+')    # High FarRed only
                else:
                    states.append('Low')        # Background/Bystander cells
                    
            R.PosLbls[pos].framelabels[t].regionprops['Cell_State'] = states

### 3. Run Spatial Statistics
Calculating Global and Local autocorrelation, Bivariate interactions, and Neighborhood Enrichment.

In [ ]:
target_channels = ['Red', 'FarRed']
biv_pairs = [('Red', 'FarRed')]

print("\n--- Running Spatial Stats Integration ---")
R.calculate_spatial_stats(
    Position=test_positions, 
    metrics=[
        'morans_i', 
        'gearys_c', 
        'neighborhood_enrichment', 
        'bivariate_moran', 
        'local_morans_i', 
        'local_bivariate_moran'
    ],
    channels=target_channels,
    bivariate_pairs=biv_pairs,
    cluster_key='Cell_State', # Uses the discrete Cell_State column we generated
    n_neighs=10,              # Uses 10 neighbors for this specific dataset  
    export_h5ad=True,         # Exports AnnData for external use
    save=False         
)

### 4. Diagnostics: View Raw Outputs & Counts

In [ ]:
print("\n--- Diagnostic: Cell Counts per Frame ---")
counts_dict = {'Frame': R.frames}
for pos in test_positions:
    counts_dict[pos] = R.PosLbls[pos].num
df_counts = pd.DataFrame(counts_dict)
print(df_counts) 

print("\n--- Viewing Raw Local Statistics (First 5 Cells) ---")
df = R.PosLbls[test_positions[0]].framelabels[0].regionprops
print(df[['mean_Red', 'local_moran_I_Red', 'local_moran_q_Red', 'local_moran_p_Red']].head())

### 5. Visualization: Generating Matplotlib Plots
Showcasing all functionality: Global metrics, grouped bar chart summaries, static X/Y spatial maps, and categorical enrichment.

In [ ]:
# Define exact colors to match the biology using the exact channel names
my_colors = {
    'Red': '#e41a1c',                         # Univariate: Red
    'FarRed': '#800080',                      # Univariate: Purple
    'Red vs FarRed': '#ff7f00',               # Bivariate: Orange
    ('Red+', 'Red+'): '#e41a1c',              # Nhood: Red - Red
    ('Red+', 'FarRed+'): '#ff7f00',           # Nhood: Red - FarRed
    ('Double+', 'Double+'): '#4daf4a',        # Nhood: Double+ - Double+ (Green)
    ('Double+', 'Low'): '#377eb8',            # Nhood: Double+ - Low (Blue)
}

# --- A. GLOBAL METRICS ---
print("\n--- Plotting Univariate: Moran's I ---")
R.plot_spatial_stats(Position=test_positions, metric='morans_i', channels=target_channels, custom_colors=my_colors)

print("\n--- Plotting Univariate: Geary's C ---")
R.plot_spatial_stats(Position=test_positions, metric='gearys_c', channels=target_channels, custom_colors=my_colors)

print("\n--- Plotting Bivariate Moran's I ---")
R.plot_spatial_stats(Position=test_positions, metric='bivariate_moran', custom_colors=my_colors)


# --- B. SUMMARY BAR CHARTS ---
# Because there is only 1 timepoint, 'summary' forces a clean grouped bar chart.
print("\n--- Plotting Expression Summary (Mean Intensity) ---")
R.plot_spatial_stats(Position=test_positions, metric='expression', channels=target_channels, plot_type='summary', custom_colors=my_colors)

print("\n--- Plotting Local Moran's I Summary (% Hotspots) ---")
R.plot_spatial_stats(Position=test_positions, metric='local_morans_i', channels=target_channels, plot_type='summary', custom_colors=my_colors)

print("\n--- Plotting Local Bivariate Moran's I Summary (% Hotspots) ---")
R.plot_spatial_stats(Position=test_positions, metric='local_bivariate_moran', plot_type='summary', custom_colors=my_colors)


# --- C. SPATIAL SNAPSHOT MAPS ---
print("\n--- Plotting Expression (Spatial Map Snapshot) ---")
R.plot_spatial_stats(Position=test_positions, metric='expression', channels=target_channels, plot_type='spatial_map', frame_idx=0)

print("\n--- Plotting Local Univariate (Spatial Map Snapshot) ---")
R.plot_spatial_stats(Position=test_positions, metric='local_morans_i', channels=target_channels, plot_type='spatial_map', frame_idx=0)

print("\n--- Plotting Local Bivariate (Spatial Map Snapshot) ---")
R.plot_spatial_stats(Position=test_positions, metric='local_bivariate_moran', plot_type='spatial_map', frame_idx=0)


# --- D. NEIGHBORHOOD ENRICHMENT ---
print("\n--- Plotting Neighborhood Enrichment Bar Chart ---")
my_pairs = [
    ('Red+', 'Red+'), 
    ('Red+', 'FarRed+'), 
    ('Double+', 'Double+'), 
    ('Double+', 'Low')
]
R.plot_spatial_stats(Position=test_positions, metric='neighborhood_enrichment', nhood_pairs=my_pairs, custom_colors=my_colors)

### 6. Interactive Napari Visualization: Stacked Metric Layers
This loads the underlying TIFF image and stacks 3 toggleable point layers (Expression, Local Moran's I, and Bivariate Moran's I) in the same viewer!

In [ ]:
print("\n--- Opening Napari to view EXPRESSION & HOTSPOTS ---")

# Call 1: Clears the viewer, and loads the Red expression map
viewer = R.show_spatial_map_napari(
    pos=test_positions[0], 
    Channel='Red',
    metric='expression', 
    frame_idx=0,  
    size=10,
    load_images=True,   # Loads the TIF because it's only 1 frame!
    clear_viewer=True   
)

# Call 2: Stacks the Red Hotspots on top of the same viewer
R.show_spatial_map_napari(
    pos=test_positions[0], 
    Channel='Red',
    metric='local_morans_i', 
    frame_idx=0,  
    size=10,
    load_images=False 
)

# Call 3: Stacks the Red vs FarRed Bivariate Hotspots on top
R.show_spatial_map_napari(
    pos=test_positions[0], 
    Channel='Red vs FarRed',
    metric='local_bivariate_moran', 
    frame_idx=0,  
    size=10,
    load_images=False 
)